In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../src")

In [3]:
PDF_PATH = pdf_path = "../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "openai/gpt-oss-20b"

SEED_ONTOLOGY_INPUT = str("../data/ontology/ContextOntology-COInd4.owl")
ONTOLOGY_PATH = "../data/ontology/ContextOntology-COInd4.owl"

# TaxoDrivenKG

In [5]:
# Main entry point for the XQuality TaxoDrivenKG adaptation as a Jupyter cell.
from __future__ import annotations

print("hi")

import sys
import datetime
from pathlib import Path
from typing import Dict, List, Tuple

# ---------------------------------------------------------
# Make local TaxoDrivenKG package importable
# ---------------------------------------------------------
TAXODRIVEN_DIR = Path("../approaches/TaxoDrivenKG").resolve()
if str(TAXODRIVEN_DIR) not in sys.path:
    sys.path.insert(0, str(TAXODRIVEN_DIR))

print("Using TaxoDrivenKG dir:", TAXODRIVEN_DIR)

from consts import PATH
from utils import load_json_file, save_json_file
from utils_LLM import InfoExtractor, TextChunker
from taxonomy import OntologyRetriever
from backends.openai_compatible import OpenAICompatibleBackend
from backends.ollama_backend import OllamaBackend


def load_translated_text_from_state(state_json_path: str | Path) -> Tuple[str, List[dict]]:
    """Load NeoOLAF layer00 state JSON and concatenate only translated_text blocks."""
    state = load_json_file(state_json_path)
    blocks = state.get("content_blocks", [])
    translated_blocks: List[dict] = []
    merged_parts: List[str] = []

    for block in blocks:
        translated_text = (block.get("translated_text") or "").strip()
        if translated_text:
            translated_blocks.append(block)
            merged_parts.append(translated_text)

    full_text = "\n\n".join(merged_parts)
    return full_text, translated_blocks


def build_backend(
    backend_name: str,
    host: str,
    api_key: str = "dummy",
    referer: str = "",
    title: str = "TaxoDrivenKG-XQuality",
):
    """Create the requested backend."""
    if backend_name == "ollama":
        return OllamaBackend(host=host)

    extra_headers = None
    if backend_name == "openrouter":
        extra_headers = {}
        if referer:
            extra_headers["HTTP-Referer"] = referer
        if title:
            extra_headers["X-Title"] = title

    return OpenAICompatibleBackend(
        base_url=host,
        api_key=api_key,
        extra_headers=extra_headers,
    )


# ---------------------------------------------------------
# User config
# ---------------------------------------------------------
STATE_JSON = "../runs/run_20260407_021434/layer00_preprocessing/Chapitre_8_Alarmes_et_messages.json"
ONTOLOGY = "../../data/ontology/ContextOntology-COInd4.owl"

BACKEND = "ollama"   # choices: "vllm", "openai", "openrouter", "ollama"
HOST = "http://localhost:11434"
API_KEY = "dummy"
MODEL_NAME = "llama3:latest"

EXPERIMENT = "base"
CHUNK_TOKENS = 600
MAX_TAXONOMY_HITS = 40

OUTPUT_DIR = PATH["outputs"]["base"]
PROMPTS_DIR = PATH["outputs"]["prompts"]
CONVERSATIONS_DIR = PATH["outputs"]["conversations"]

REFERER = ""
TITLE = "TaxoDrivenKG-XQuality"


# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
output_dir = Path(OUTPUT_DIR)
prompts_dir = Path(PROMPTS_DIR)
conversations_dir = Path(CONVERSATIONS_DIR)

output_dir.mkdir(parents=True, exist_ok=True)
prompts_dir.mkdir(parents=True, exist_ok=True)
conversations_dir.mkdir(parents=True, exist_ok=True)

start_time = datetime.datetime.now()
print(f"Time now: {start_time}")
print(f"Loading state JSON: {STATE_JSON}")

full_text, translated_blocks = load_translated_text_from_state(STATE_JSON)
print(f"Translated blocks loaded: {len(translated_blocks)}")
print(f"Merged translated chars: {len(full_text)}")

backend = build_backend(
    backend_name=BACKEND,
    host=HOST,
    api_key=API_KEY,
    referer=REFERER,
    title=TITLE,
)

model = InfoExtractor(
    backend=backend,
    model_name=MODEL_NAME,
    exp=EXPERIMENT,
    n_shot=3,
)

chunker = TextChunker(text_max_tokens=CHUNK_TOKENS)
retriever = OntologyRetriever(ONTOLOGY)

text_chunks, start_idxs = chunker.get_text_chunks(full_text)

outputs: Dict[str, dict] = {}
conversations: Dict[str, list] = {}
prompts: Dict[str, str] = {}

for chunk_i, (text, start_idx) in enumerate(zip(text_chunks, start_idxs), start=1):
    end_idx = start_idx + len(text)
    print(f"Chunk [{chunk_i}/{len(text_chunks)}] Retrieving ontology candidates")
    retrieved_nodes = retriever.retrieve(text, max_hits=MAX_TAXONOMY_HITS)

    print(f"Chunk [{chunk_i}/{len(text_chunks)}] Running backend")
    output, conversation = model.run(text, retrieved_nodes)

    span_key = str((start_idx, end_idx))
    outputs[span_key] = output
    conversations[span_key] = conversation
    prompts[span_key] = conversation[0]["content"]

state_name = Path(STATE_JSON).stem

save_json_file(output_dir / f"{state_name}.json", outputs)
save_json_file(prompts_dir / f"{state_name}.json", prompts)
save_json_file(conversations_dir / f"{state_name}.json", conversations)

print(f"Saved outputs to: {output_dir / f'{state_name}.json'}")
print(f"Saved prompts to: {prompts_dir / f'{state_name}.json'}")
print(f"Saved conversations to: {conversations_dir / f'{state_name}.json'}")
print(f"Time elapsed: {datetime.datetime.now() - start_time}")
print("Done.")

hi
Using TaxoDrivenKG dir: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG


c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Time now: 2026-04-07 02:43:36.772210
Loading state JSON: ../runs/run_20260407_021434/layer00_preprocessing/Chapitre_8_Alarmes_et_messages.json
Translated blocks loaded: 113
Merged translated chars: 66918
Chunk [1/30] Retrieving ontology candidates
Chunk [1/30] Running backend
Chunk [2/30] Retrieving ontology candidates
Chunk [2/30] Running backend
Chunk [3/30] Retrieving ontology candidates
Chunk [3/30] Running backend
Chunk [4/30] Retrieving ontology candidates
Chunk [4/30] Running backend
Chunk [5/30] Retrieving ontology candidates
Chunk [5/30] Running backend
Chunk [6/30] Retrieving ontology candidates
Chunk [6/30] Running backend
Chunk [7/30] Retrieving ontology candidates
Chunk [7/30] Running backend
Chunk [8/30] Retrieving ontology candidates
Chunk [8/30] Running backend
Chunk [9/30] Retrieving ontology candidates
Chunk [9/30] Running backend
Chunk [10/30] Retrieving ontology candidates
Chunk [10/30] Running backend
Chunk [11/30] Retrieving ontology candidates
Chunk [11/30] Runni

In [9]:
from __future__ import annotations

print("hi")

import os
import re
import sys
import json
import datetime
from pathlib import Path
from typing import Dict, List, Tuple
from dataclasses import dataclass

# ---------------------------------------------------------
# User paths / config
# ---------------------------------------------------------
TAXODRIVEN_DIR = Path("../approaches/TaxoDrivenKG").resolve()
ENV_PATH = Path("../../.env").resolve()

STATE_JSON = "../runs/run_20260407_021434/layer00_preprocessing/Chapitre_8_Alarmes_et_messages.json"
ONTOLOGY = "../../data/ontology/ContextOntology-COInd4.owl"

BACKEND = "openrouter"   # choices: vllm, openai, openrouter, ollama
HOST = "https://openrouter.ai/api/v1"   # ollama: http://localhost:11434 ; vllm: http://localhost:8000/v1
API_KEY = ""   # leave empty to load from .env
MODEL_NAME = "openai/gpt-oss-20b"       # e.g. llama3:latest for ollama

EXPERIMENT = "base"
CHUNK_TOKENS = 600
MAX_TAXONOMY_HITS = 40
N_SHOT = 3

REFERER = "http://localhost"
TITLE = "NeoOLAF-TaxoDrivenKG-Comparison"

OUTPUT_DIR = TAXODRIVEN_DIR / "outputs"
PROMPTS_DIR = TAXODRIVEN_DIR / "outputs_prompts"
CONVERSATIONS_DIR = TAXODRIVEN_DIR / "conversations"
TTL_DIR = TAXODRIVEN_DIR / "outputs_ttl"
FEW_SHOT_PATH = TAXODRIVEN_DIR / "few_shot.json"

# ---------------------------------------------------------
# Local import setup
# ---------------------------------------------------------
if str(TAXODRIVEN_DIR) not in sys.path:
    sys.path.insert(0, str(TAXODRIVEN_DIR))

# ---------------------------------------------------------
# Optional .env loading
# ---------------------------------------------------------
def load_dotenv_file(env_path: Path) -> None:
    """Minimal .env loader to avoid dependency issues."""
    if not env_path.exists():
        print(f".env not found at: {env_path}")
        return
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

load_dotenv_file(ENV_PATH)

# ---------------------------------------------------------
# Imports from TaxoDrivenKG repo
# ---------------------------------------------------------
from utils import load_json_file, save_json_file
from utils_LLM import TextChunker, delimiters
from prompt_templates import PROMPT_TEMPLATE, PROMPT_TEMPLATE_ZERO_SHOT

# ---------------------------------------------------------
# Backend wrappers
# ---------------------------------------------------------
class OpenAICompatibleBackend:
    """Simple wrapper around OpenAI-compatible chat completion APIs."""
    def __init__(
        self,
        base_url: str,
        api_key: str,
        extra_headers: Dict[str, str] | None = None,
        timeout: float = 300.0,
    ) -> None:
        from openai import OpenAI
        self.client = OpenAI(
            base_url=base_url,
            api_key=api_key,
            timeout=timeout,
            default_headers=extra_headers or {},
        )

    def chat(
        self,
        model_name: str,
        messages: List[Dict[str, str]],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> str:
        response = self.client.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content or ""


class OllamaBackend:
    """Simple Ollama wrapper."""
    def __init__(self, host: str = "http://localhost:11434", timeout: float = 300.0) -> None:
        self.host = host.rstrip("/")
        self.timeout = timeout

    def chat(
        self,
        model_name: str,
        messages: List[Dict[str, str]],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> str:
        import requests

        payload = {
            "model": model_name,
            "messages": messages,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": max_tokens,
            },
        }
        response = requests.post(
            f"{self.host}/api/chat",
            json=payload,
            timeout=self.timeout,
        )
        response.raise_for_status()
        data = response.json()
        return data.get("message", {}).get("content", "")

# ---------------------------------------------------------
# Ontology retrieval
# ---------------------------------------------------------
from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS
from rdflib.namespace import OWL

@dataclass
class OntologyEntry:
    uri: str
    label: str
    entry_type: str  # class / property

class OntologyRetriever:
    """Harvest labels from OWL and retrieve lexical matches."""
    def __init__(self, ontology_path: str | Path) -> None:
        self.ontology_path = str(ontology_path)
        self.graph = Graph()
        self.graph.parse(self.ontology_path)
        self.entries = self._harvest_entries()

    def _local_name(self, uri: URIRef) -> str:
        value = str(uri)
        if "#" in value:
            return value.split("#")[-1]
        return value.rstrip("/").split("/")[-1]

    def _harvest_entries(self) -> List[OntologyEntry]:
        entries: List[OntologyEntry] = []
        seen = set()

        def add_entry(uri: URIRef, label: str, entry_type: str) -> None:
            key = (str(uri), label.strip().lower(), entry_type)
            if key in seen:
                return
            seen.add(key)
            entries.append(OntologyEntry(str(uri), label.strip(), entry_type))

        for subject in set(self.graph.subjects(RDF.type, OWL.Class)):
            labels = list(self.graph.objects(subject, RDFS.label))
            if labels:
                for lbl in labels:
                    add_entry(subject, str(lbl), "class")
            else:
                add_entry(subject, self._local_name(subject), "class")

        for predicate_type in [OWL.ObjectProperty, OWL.DatatypeProperty, RDF.Property]:
            for subject in set(self.graph.subjects(RDF.type, predicate_type)):
                labels = list(self.graph.objects(subject, RDFS.label))
                if labels:
                    for lbl in labels:
                        add_entry(subject, str(lbl), "property")
                else:
                    add_entry(subject, self._local_name(subject), "property")

        return entries

    def retrieve(self, text: str, max_hits: int = 40) -> Dict[str, dict]:
        lowered = text.lower()
        hits: Dict[str, dict] = {}
        for entry in self.entries:
            if entry.label.lower() in lowered:
                hits[entry.label] = {
                    "uri": entry.uri,
                    "label": entry.label,
                    "type": entry.entry_type,
                    "score": 1.0,
                }
                if len(hits) >= max_hits:
                    break
        return hits

# ---------------------------------------------------------
# Prompted extractor kept close to TaxoDrivenKG
# ---------------------------------------------------------
class InfoExtractor:
    """TaxoDriven-style extractor with pluggable backend."""
    def __init__(
        self,
        backend,
        model_name: str,
        exp: str = "base",
        n_shot: int = 3,
        few_shot_path: Path | None = None,
    ) -> None:
        self.backend = backend
        self.model_name = model_name
        self.exp = exp

        if exp == "0_shot":
            self.prompt_template = PROMPT_TEMPLATE_ZERO_SHOT
        else:
            self.prompt_template = PROMPT_TEMPLATE

        self.formatted_examples = ""
        if few_shot_path and few_shot_path.exists():
            few_shot_examples = json.loads(few_shot_path.read_text(encoding="utf-8"))
            for i, example in enumerate(few_shot_examples[:n_shot]):
                self.formatted_examples += f"\nExample {i+1}:\n{example}"

    def parse_response(self, response: str, with_description: bool = True) -> dict:
        out = {"entities": [], "relationships": []}

        start_index = response.find('("')
        if start_index > 0:
            response = response[start_index:]

        response = response.split(delimiters["completion_delimiter"])[0]
        response = response.split(delimiters["record_delimiter"])
        response = [
            r.lstrip("\n").rstrip("\n").lstrip("(").rstrip(")")
            for r in response
        ]

        pattern = r"<\s*\|\s*>"
        response = [re.split(pattern, r) for r in response]

        for r in response:
            if not r or len(r) == 0:
                continue

            if "entity" in r[0]:
                if with_description and len(r) == 4:
                    out["entities"].append(
                        {"name": r[1], "label": r[2], "description": r[3]}
                    )
                elif (not with_description) and len(r) == 3:
                    out["entities"].append(
                        {"name": r[1], "label": r[2]}
                    )

            elif "relationship" in r[0]:
                if len(r) == 4:
                    out["relationships"].append(
                        {"source": r[1], "target": r[2], "relation": r[3]}
                    )
        return out

    def run(self, text: str, retrieved_nodes: Dict[str, dict]) -> Tuple[dict, list]:
        potential_entities = ", ".join(list(retrieved_nodes.keys()))

        prompt = self.prompt_template.format(
            **delimiters,
            formatted_examples=self.formatted_examples,
            input_text=text.replace("{", "").replace("}", ""),
            potential_entities=potential_entities,
        ).format(**delimiters)

        conversation = [{"role": "user", "content": prompt}]

        pred_content = self.backend.chat(
            model_name=self.model_name,
            messages=conversation,
            temperature=0.0,
            max_tokens=2048,
        )

        conversation.append({"role": "assistant", "content": pred_content})
        parsed = self.parse_response(pred_content)
        return parsed, conversation

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def load_translated_text_from_state(state_json_path: str | Path) -> Tuple[str, List[dict]]:
    """Load NeoOLAF layer00 JSON and concatenate only translated_text blocks."""
    state = load_json_file(state_json_path)
    blocks = state.get("content_blocks", [])
    translated_blocks: List[dict] = []
    merged_parts: List[str] = []

    for block in blocks:
        translated_text = (block.get("translated_text") or "").strip()
        if translated_text:
            translated_blocks.append(block)
            merged_parts.append(translated_text)

    full_text = "\n\n".join(merged_parts)
    return full_text, translated_blocks

def build_backend(
    backend_name: str,
    host: str,
    api_key: str = "",
    referer: str = "",
    title: str = "TaxoDrivenKG-XQuality",
):
    """Create the requested backend like run.py."""
    if backend_name == "ollama":
        return OllamaBackend(host=host)

    resolved_api_key = api_key
    if not resolved_api_key:
        if backend_name == "openrouter":
            resolved_api_key = os.getenv("OPENROUTER_API_KEY", "")
        elif backend_name == "openai":
            resolved_api_key = os.getenv("OPENAI_API_KEY", "")
        else:
            resolved_api_key = os.getenv("OPENAI_API_KEY", "dummy")

    extra_headers = None
    if backend_name == "openrouter":
        extra_headers = {}
        if referer:
            extra_headers["HTTP-Referer"] = referer
        if title:
            extra_headers["X-Title"] = title

    return OpenAICompatibleBackend(
        base_url=host,
        api_key=resolved_api_key,
        extra_headers=extra_headers,
    )

# ---------------------------------------------------------
# TTL serialization
# ---------------------------------------------------------
EX = Namespace("http://taxodrivenkg-xquality.org/resource/")
VOC = Namespace("http://taxodrivenkg-xquality.org/vocab/")

def _safe_name(text: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9_]+", "_", text.strip())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned or "Unnamed"

def build_result_ontology_graph(seed_ontology_path: str | Path, outputs: Dict[str, dict]) -> Graph:
    """Original ontology + extracted labels/relations as ontology additions."""
    graph = Graph()
    graph.parse(str(seed_ontology_path))

    graph.bind("owl", OWL)
    graph.bind("rdfs", RDFS)
    graph.bind("rdf", RDF)
    graph.bind("ex", EX)
    graph.bind("voc", VOC)

    added_classes = set()
    added_properties = set()

    for _, chunk_output in outputs.items():
        for entity in chunk_output.get("entities", []):
            label = entity.get("label", "").strip()
            if not label:
                continue
            uri = VOC[_safe_name(label)]
            if uri not in added_classes:
                graph.add((uri, RDF.type, OWL.Class))
                graph.add((uri, RDFS.label, Literal(label)))
                added_classes.add(uri)

        for relation in chunk_output.get("relationships", []):
            rel_label = relation.get("relation", "").strip()
            if not rel_label:
                continue
            uri = VOC[_safe_name(rel_label)]
            if uri not in added_properties:
                graph.add((uri, RDF.type, OWL.ObjectProperty))
                graph.add((uri, RDFS.label, Literal(rel_label)))
                added_properties.add(uri)

    return graph

def build_result_kg_graph(outputs: Dict[str, dict]) -> Graph:
    """KG triples from extracted entities and relationships."""
    graph = Graph()
    graph.bind("rdf", RDF)
    graph.bind("rdfs", RDFS)
    graph.bind("ex", EX)
    graph.bind("voc", VOC)

    for span_key, chunk_output in outputs.items():
        chunk_node = EX[f"chunk_{_safe_name(span_key)}"]

        for entity in chunk_output.get("entities", []):
            name = entity.get("name", "").strip()
            label = entity.get("label", "").strip()
            description = entity.get("description", "").strip()

            if not name:
                continue

            entity_uri = EX[_safe_name(name)]
            type_uri = VOC[_safe_name(label or "Entity")]

            graph.add((entity_uri, RDF.type, type_uri))
            graph.add((entity_uri, RDFS.label, Literal(name)))
            if description:
                graph.add((entity_uri, RDFS.comment, Literal(description)))
            graph.add((entity_uri, VOC.extractedFromChunk, chunk_node))

        for relation in chunk_output.get("relationships", []):
            source_name = relation.get("source", "").strip()
            target_name = relation.get("target", "").strip()
            rel_label = relation.get("relation", "").strip()

            if not source_name or not target_name or not rel_label:
                continue

            source_uri = EX[_safe_name(source_name)]
            target_uri = EX[_safe_name(target_name)]
            rel_uri = VOC[_safe_name(rel_label)]

            graph.add((source_uri, rel_uri, target_uri))

    return graph

def save_ttl_outputs(outputs: Dict[str, dict], seed_ontology_path: str | Path, ttl_dir: Path, state_name: str):
    ttl_dir.mkdir(parents=True, exist_ok=True)

    ontology_graph = build_result_ontology_graph(seed_ontology_path, outputs)
    kg_graph = build_result_kg_graph(outputs)

    ontology_path = ttl_dir / f"{state_name}_result_ontology.ttl"
    kg_path = ttl_dir / f"{state_name}_result_kg.ttl"

    ontology_graph.serialize(destination=str(ontology_path), format="turtle")
    kg_graph.serialize(destination=str(kg_path), format="turtle")

    return ontology_path, kg_path

# ---------------------------------------------------------
# Run block: same operations as run.py
# ---------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
CONVERSATIONS_DIR.mkdir(parents=True, exist_ok=True)
TTL_DIR.mkdir(parents=True, exist_ok=True)

start_time = datetime.datetime.now()
print(f"Time now: {start_time}")
print(f"Loading state JSON: {STATE_JSON}")
print(f"Using backend: {BACKEND}")
print(f"Using model: {MODEL_NAME}")
print(f"Using ontology: {ONTOLOGY}")
print(f"Using .env: {ENV_PATH}")

full_text, translated_blocks = load_translated_text_from_state(STATE_JSON)
print(f"Translated blocks loaded: {len(translated_blocks)}")
print(f"Merged translated chars: {len(full_text)}")

backend = build_backend(
    backend_name=BACKEND,
    host=HOST,
    api_key=API_KEY,
    referer=REFERER,
    title=TITLE,
)

model = InfoExtractor(
    backend=backend,
    model_name=MODEL_NAME,
    exp=EXPERIMENT,
    n_shot=N_SHOT,
    few_shot_path=FEW_SHOT_PATH,
)

chunker = TextChunker(text_max_tokens=CHUNK_TOKENS)
retriever = OntologyRetriever(ONTOLOGY)

text_chunks, start_idxs = chunker.get_text_chunks(full_text)

outputs: Dict[str, dict] = {}
conversations: Dict[str, list] = {}
prompts: Dict[str, str] = {}

for chunk_i, (text, start_idx) in enumerate(zip(text_chunks, start_idxs), start=1):
    end_idx = start_idx + len(text)

    print(f"Chunk [{chunk_i}/{len(text_chunks)}] Retrieving ontology candidates")
    retrieved_nodes = retriever.retrieve(text, max_hits=MAX_TAXONOMY_HITS)

    print(f"Chunk [{chunk_i}/{len(text_chunks)}] Running backend")
    output, conversation = model.run(text, retrieved_nodes)

    span_key = str((start_idx, end_idx))
    outputs[span_key] = output
    conversations[span_key] = conversation
    prompts[span_key] = conversation[0]["content"]

state_name = Path(STATE_JSON).stem

output_json_path = OUTPUT_DIR / f"{state_name}.json"
prompts_json_path = PROMPTS_DIR / f"{state_name}.json"
conversations_json_path = CONVERSATIONS_DIR / f"{state_name}.json"

save_json_file(output_json_path, outputs)
save_json_file(prompts_json_path, prompts)
save_json_file(conversations_json_path, conversations)

ontology_ttl_path, kg_ttl_path = save_ttl_outputs(
    outputs=outputs,
    seed_ontology_path=ONTOLOGY,
    ttl_dir=TTL_DIR,
    state_name=state_name,
)

print(f"Saved outputs to: {output_json_path}")
print(f"Saved prompts to: {prompts_json_path}")
print(f"Saved conversations to: {conversations_json_path}")
print(f"Saved ontology TTL to: {ontology_ttl_path}")
print(f"Saved KG TTL to: {kg_ttl_path}")
print(f"Time elapsed: {datetime.datetime.now() - start_time}")
print("Done.")

hi
Time now: 2026-04-07 03:00:22.336035
Loading state JSON: ../runs/run_20260407_021434/layer00_preprocessing/Chapitre_8_Alarmes_et_messages.json
Using backend: openrouter
Using model: openai/gpt-oss-20b
Using ontology: ../../data/ontology/ContextOntology-COInd4.owl
Using .env: C:\Users\henri\Documents\git\post-doc\NeoOLAF\.env
Translated blocks loaded: 113
Merged translated chars: 66918
Chunk [1/30] Retrieving ontology candidates
Chunk [1/30] Running backend
Chunk [2/30] Retrieving ontology candidates
Chunk [2/30] Running backend
Chunk [3/30] Retrieving ontology candidates
Chunk [3/30] Running backend
Chunk [4/30] Retrieving ontology candidates
Chunk [4/30] Running backend
Chunk [5/30] Retrieving ontology candidates
Chunk [5/30] Running backend
Chunk [6/30] Retrieving ontology candidates
Chunk [6/30] Running backend
Chunk [7/30] Retrieving ontology candidates
Chunk [7/30] Running backend
Chunk [8/30] Retrieving ontology candidates
Chunk [8/30] Running backend
Chunk [9/30] Retrieving o

In [ ]:
# Main entry point for the XQuality TaxoDrivenKG adaptation as a Jupyter cell.
from __future__ import annotations

print("hi")

import sys
import datetime
from pathlib import Path
from typing import Dict, List, Tuple

# ---------------------------------------------------------
# Make local TaxoDrivenKG package importable
# ---------------------------------------------------------
TAXODRIVEN_DIR = Path("../approaches/TaxoDrivenKG").resolve()
if str(TAXODRIVEN_DIR) not in sys.path:
    sys.path.insert(0, str(TAXODRIVEN_DIR))

print("Using TaxoDrivenKG dir:", TAXODRIVEN_DIR)

from consts import PATH
from utils import load_json_file, save_json_file
from utils_LLM import InfoExtractor, TextChunker
from taxonomy import OntologyRetriever
from backends.openai_compatible import OpenAICompatibleBackend
from backends.ollama_backend import OllamaBackend


def load_translated_text_from_state(state_json_path: str | Path) -> Tuple[str, List[dict]]:
    """Load NeoOLAF layer00 state JSON and concatenate only translated_text blocks."""
    state = load_json_file(state_json_path)
    blocks = state.get("content_blocks", [])
    translated_blocks: List[dict] = []
    merged_parts: List[str] = []

    for block in blocks:
        translated_text = (block.get("translated_text") or "").strip()
        if translated_text:
            translated_blocks.append(block)
            merged_parts.append(translated_text)

    full_text = "\n\n".join(merged_parts)
    return full_text, translated_blocks


def build_backend(
    backend_name: str,
    host: str,
    api_key: str = "dummy",
    referer: str = "",
    title: str = "TaxoDrivenKG-XQuality",
):
    """Create the requested backend."""
    if backend_name == "ollama":
        return OllamaBackend(host=host)

    extra_headers = None
    if backend_name == "openrouter":
        extra_headers = {}
        if referer:
            extra_headers["HTTP-Referer"] = referer
        if title:
            extra_headers["X-Title"] = title

    return OpenAICompatibleBackend(
        base_url=host,
        api_key=api_key,
        extra_headers=extra_headers,
    )


# ---------------------------------------------------------
# User config
# ---------------------------------------------------------
STATE_JSON = "../runs/run_20260407_021434/layer00_preprocessing/Chapitre_8_Alarmes_et_messages.json"
ONTOLOGY = "../../data/ontology/ContextOntology-COInd4.owl"

BACKEND = "ollama"   # choices: "vllm", "openai", "openrouter", "ollama"
HOST = "http://localhost:11434"
API_KEY = "dummy"
MODEL_NAME = "llama3:latest"

EXPERIMENT = "base"
CHUNK_TOKENS = 600
MAX_TAXONOMY_HITS = 40

OUTPUT_DIR = PATH["outputs"]["base"]
PROMPTS_DIR = PATH["outputs"]["prompts"]
CONVERSATIONS_DIR = PATH["outputs"]["conversations"]

REFERER = ""
TITLE = "TaxoDrivenKG-XQuality"


# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
output_dir = Path(OUTPUT_DIR)
prompts_dir = Path(PROMPTS_DIR)
conversations_dir = Path(CONVERSATIONS_DIR)

output_dir.mkdir(parents=True, exist_ok=True)
prompts_dir.mkdir(parents=True, exist_ok=True)
conversations_dir.mkdir(parents=True, exist_ok=True)

start_time = datetime.datetime.now()
print(f"Time now: {start_time}")
print(f"Loading state JSON: {STATE_JSON}")

full_text, translated_blocks = load_translated_text_from_state(STATE_JSON)
print(f"Translated blocks loaded: {len(translated_blocks)}")
print(f"Merged translated chars: {len(full_text)}")

backend = build_backend(
    backend_name=BACKEND,
    host=HOST,
    api_key=API_KEY,
    referer=REFERER,
    title=TITLE,
)

model = InfoExtractor(
    backend=backend,
    model_name=MODEL_NAME,
    exp=EXPERIMENT,
    n_shot=3,
)

chunker = TextChunker(text_max_tokens=CHUNK_TOKENS)
retriever = OntologyRetriever(ONTOLOGY)

text_chunks, start_idxs = chunker.get_text_chunks(full_text)

outputs: Dict[str, dict] = {}
conversations: Dict[str, list] = {}
prompts: Dict[str, str] = {}

for chunk_i, (text, start_idx) in enumerate(zip(text_chunks, start_idxs), start=1):
    end_idx = start_idx + len(text)
    print(f"Chunk [{chunk_i}/{len(text_chunks)}] Retrieving ontology candidates")
    retrieved_nodes = retriever.retrieve(text, max_hits=MAX_TAXONOMY_HITS)

    print(f"Chunk [{chunk_i}/{len(text_chunks)}] Running backend")
    output, conversation = model.run(text, retrieved_nodes)

    span_key = str((start_idx, end_idx))
    outputs[span_key] = output
    conversations[span_key] = conversation
    prompts[span_key] = conversation[0]["content"]

state_name = Path(STATE_JSON).stem

save_json_file(output_dir / f"{state_name}.json", outputs)
save_json_file(prompts_dir / f"{state_name}.json", prompts)
save_json_file(conversations_dir / f"{state_name}.json", conversations)

print(f"Saved outputs to: {output_dir / f'{state_name}.json'}")
print(f"Saved prompts to: {prompts_dir / f'{state_name}.json'}")
print(f"Saved conversations to: {conversations_dir / f'{state_name}.json'}")
print(f"Time elapsed: {datetime.datetime.now() - start_time}")
print("Done.")

hi
Using TaxoDrivenKG dir: C:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\approaches\TaxoDrivenKG


c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Time now: 2026-04-07 02:43:36.772210
Loading state JSON: ../runs/run_20260407_021434/layer00_preprocessing/Chapitre_8_Alarmes_et_messages.json
Translated blocks loaded: 113
Merged translated chars: 66918
Chunk [1/30] Retrieving ontology candidates
Chunk [1/30] Running backend
Chunk [2/30] Retrieving ontology candidates
Chunk [2/30] Running backend
Chunk [3/30] Retrieving ontology candidates
Chunk [3/30] Running backend
Chunk [4/30] Retrieving ontology candidates
Chunk [4/30] Running backend
Chunk [5/30] Retrieving ontology candidates
Chunk [5/30] Running backend
Chunk [6/30] Retrieving ontology candidates
Chunk [6/30] Running backend
Chunk [7/30] Retrieving ontology candidates
Chunk [7/30] Running backend
Chunk [8/30] Retrieving ontology candidates
Chunk [8/30] Running backend
Chunk [9/30] Retrieving ontology candidates
Chunk [9/30] Running backend
Chunk [10/30] Retrieving ontology candidates
Chunk [10/30] Running backend
Chunk [11/30] Retrieving ontology candidates
Chunk [11/30] Runni

# NeoOLAF

In [21]:
import sys
sys.path.append("../../src")

In [22]:
PDF_PATH = pdf_path = "../../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
MODEL_NAME = model_name = "openai/gpt-oss-20b"

# Ontology modes:
#   None       -> no ontology
#   "minimal"  -> minimal fallback ontology
#   path       -> real ontology file
SEED_ONTOLOGY_INPUT = str("../../data/ontology/ContextOntology-COInd4.owl")
ONTOLOGY_PATH = "../../data/ontology/ContextOntology-COInd4.owl"

In [ ]:
# ---------------------------------------------------------
# Fresh NeoOLAF run from scratch with OpenRouter GPT-OSS-20B
# ---------------------------------------------------------
import os
import sys
import json
from pathlib import Path
from pprint import pprint

# ---------------------------------------------------------
# Path setup
# ---------------------------------------------------------
PROJECT_ROOT = Path("../..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

# ---------------------------------------------------------
# Minimal .env loader
# ---------------------------------------------------------
def load_dotenv_file(env_path: Path) -> None:
    if not env_path.exists():
        print(f".env not found at: {env_path}")
        return
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv_file(ENV_PATH)

# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from neoolaf.core.pipeline import Pipeline
from neoolaf.core.pipeline_state import PipelineState
from neoolaf.core.runner import Runner
from neoolaf.core.execution_config import ExecutionConfig

from neoolaf.domain.documents import Document
from neoolaf.domain.user_guidance import (
    UserGuidance,
    TypingExample,
    RelationExample,
    PromotionExample,
    NegativeExample,
)

from neoolaf.preprocessing.pdf_parsing import extract_text_from_pdf

from neoolaf.resources.llm_backends.openai_backend import OpenAIBackend
from neoolaf.resources.translation.deep_translator_backend import DeepTranslatorBackend

from neoolaf.resources.knowledge_sources.wordnet_source import WordNetSource
from neoolaf.resources.knowledge_sources.wikipedia_source import WikipediaSource
from neoolaf.resources.knowledge_sources.wikidata_source import WikidataSource
from neoolaf.resources.knowledge_sources.web_search_source import WebSearchSource

from neoolaf.ontology.factory import build_seed_ontology

from neoolaf.grounding.rag.registry import RetrievalRegistry
from neoolaf.grounding.rag.engine import SemanticRAGEngine
from neoolaf.grounding.rag.adapters.neoolaf_semantic_rag_adapter import NeoOLAFSemanticRAGAdapter
from neoolaf.grounding.rag.spaces.ontology_space import OntologySpace
from neoolaf.grounding.rag.spaces.artifact_space import ArtifactSpace
from neoolaf.grounding.rag.spaces.web_space import WebSpace
from neoolaf.grounding.rag.spaces.wikidata_space import WikidataSpace
from neoolaf.grounding.rag.spaces.wikipedia_space import WikipediaSpace
from neoolaf.grounding.rag.spaces.wordnet_space import WordNetSpace

from neoolaf.layers.layer00_preprocessing.component import PreprocessingLayer
from neoolaf.layers.layer01_linguistic_expression_extraction.component import LinguisticExpressionExtractionLayer
from neoolaf.layers.layer02_candidate_enrichment.component import CandidateEnrichmentLayer
from neoolaf.layers.layer03_candidate_typing_resolution.component import CandidateTypingResolutionLayer
from neoolaf.layers.layer04_candidate_relation_extraction.component import CandidateRelationExtractionLayer
from neoolaf.layers.layer05_candidate_triple_generation.component import CandidateTripleGenerationLayer
from neoolaf.layers.layer06_concept_relation_induction.component import ConceptRelationInductionLayer
from neoolaf.layers.layer07_hierarchisation.component import HierarchisationLayer
from neoolaf.layers.layer08_axiom_schemata_extraction.component import AxiomSchemataExtractionLayer
from neoolaf.layers.layer09_general_axiom_extraction.component import GeneralAxiomExtractionLayer
from neoolaf.layers.layer10_validation_reasoning.component import ValidationReasoningLayer
from neoolaf.layers.layer11_inference_completion.component import InferenceCompletionLayer
from neoolaf.layers.layer12_serialization.component import SerializationLayer

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
PDF_PATH = "../../data/XQuality/Textual/Chapitre_8_Alarmes_et_messages.pdf"
ONTOLOGY_PATH = "../../data/ontology/ContextOntology-COInd4.owl"
FEW_SHOTS_PATH = "../../data/XQuality/Examples/few_shots.json"

MODEL_NAME = "openai/gpt-oss-20b"

# OpenRouter host for the OpenAIBackend that appends /v1/chat/completions
OPENROUTER_HOST = "https://openrouter.ai/api"
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY", "")

if not OPENROUTER_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in environment or .env file.")

# Robust but still fast defaults
CHUNK_SIZE = 5000
CHUNK_OVERLAP = 0
MAX_CHUNKS = None
MAX_EXPRESSIONS = None
MAX_RELATION_MENTIONS = None

USE_WEB_SEARCH = False   # faster; turn True only if you explicitly want web grounding
TRANSLATE_TO_ENGLISH = True
CONTINUE_FROM_LAST = False  # fresh run from scratch

# ---------------------------------------------------------
# Load document
# ---------------------------------------------------------
raw_text = extract_text_from_pdf(str(PDF_PATH))

document = Document(
    doc_id="xquality_alarm_doc_001",
    source_path=str(PDF_PATH),
    raw_text=raw_text,
)

# ---------------------------------------------------------
# Load few-shots and project them into guidance
# ---------------------------------------------------------
few_shots = []
few_shots_path = Path(FEW_SHOTS_PATH)
if few_shots_path.exists():
    with open(few_shots_path, "r", encoding="utf-8") as f:
        few_shots = json.load(f)

typing_examples = [
    TypingExample(text="bearing", expected_type="entity"),
    TypingExample(text="overheating", expected_type="event"),
    TypingExample(text="emitted by", expected_type="relation"),
]

relation_examples = [
    RelationExample(
        text="The alarm is emitted by the PLC.",
        source_label="alarm",
        relation_label="emittedBy",
        target_label="PLC",
    ),
]

promotion_examples = [
    PromotionExample(text="overheating", promote=True, promoted_label="OverheatingEvent"),
    PromotionExample(text="emitted by", promote=True, promoted_label="emittedBy"),
]

# Convert a subset of your few-shot outputs into structured guidance
# to keep prompting useful but not too bloated.
for example in few_shots[:3]:
    alarm_label = example.get("alarm_label", "")
    if alarm_label:
        typing_examples.append(TypingExample(text=alarm_label, expected_type="event"))
        promotion_examples.append(
            PromotionExample(text=alarm_label, promote=True, promoted_label=alarm_label.title().replace(" ", ""))
        )

    for triple in example.get("triples", [])[:6]:
        src = triple.get("node_1", "")
        rel = triple.get("relation", "")
        tgt = triple.get("node_2", "")
        if src and rel and tgt:
            relation_examples.append(
                RelationExample(
                    text=f"{src} {rel} {tgt}",
                    source_label=src,
                    relation_label=rel,
                    target_label=tgt,
                )
            )

negative_examples = [
    NegativeExample(text="section 8.1", explanation="Not a semantic concept.", target_layer="layer01"),
    NegativeExample(text="page 5", explanation="Page number alone is not a semantic concept.", target_layer="layer01"),
]

# ---------------------------------------------------------
# User guidance
# ---------------------------------------------------------
guidance = UserGuidance(
    domain_focus="industrial maintenance, machine alarms, failure chains, causes, effects, interventions, and responsible actors",
    abstraction_level=(
        "Treat alarm types, failure types, intervention types, and machine-related categories as concepts; "
        "treat document-specific alarm instances, page references, and concrete occurrences as individuals."
    ),
    priority_relations=[
        "TRIGGERS",
        "CAUSES",
        "REQUIRES",
        "HANDLED_BY",
        "REFERENCES",
        "part-of",
        "affects",
        "observed-by",
        "temporal",
    ],
    population_policy=(
        "Promote a candidate to concept only if it is semantically stable and recurrent across documents; "
        "keep one-off alarm mentions and page references as individuals."
    ),
    event_modeling_preference=(
        "Treat alarms, failures, degradations, shutdowns, and anomaly states as events/states rather than simple entities."
    ),
    ontology_depth="deep",
    typing_examples=typing_examples,
    relation_examples=relation_examples,
    promotion_examples=promotion_examples,
    negative_examples=negative_examples,
)

# ---------------------------------------------------------
# Seed ontology
# ---------------------------------------------------------
seed_ontology = build_seed_ontology(str(ONTOLOGY_PATH))

# ---------------------------------------------------------
# Pipeline state
# ---------------------------------------------------------
state = PipelineState(
    document=document,
    llm_model=MODEL_NAME,
    user_guidance=guidance,
    seed_ontology=seed_ontology,
)

# ---------------------------------------------------------
# Backends
# ---------------------------------------------------------
openrouter_backend = OpenAIBackend(
    host=OPENROUTER_HOST,
    api_key=OPENROUTER_KEY,
    timeout=900,
    max_retries=3,
    retry_wait_seconds=5.0,
    referer="http://localhost",
    title="NeoOLAF-XQuality",
)

translator = DeepTranslatorBackend(max_chars=4000)

wordnet_source = WordNetSource()
wikipedia_source = WikipediaSource(language="en")
wikidata_source = WikidataSource(language="en")
web_search_source = WebSearchSource()

# ---------------------------------------------------------
# Build SemanticRAG registry and adapter
# ---------------------------------------------------------
registry = RetrievalRegistry()
registry.register(OntologySpace(state.seed_ontology))
registry.register(ArtifactSpace(state))
registry.register(WikidataSpace(wikidata_source))
registry.register(WikipediaSpace(wikipedia_source))
registry.register(WordNetSpace(wordnet_source))

# Keep web search optional because it slows runs a lot
if USE_WEB_SEARCH:
    registry.register(WebSpace(web_search_source))

semantic_rag_engine = SemanticRAGEngine(
    registry=registry,
    ollama_backend=openrouter_backend,  # kept for compatibility with current NeoOLAF code signature
    model_name=MODEL_NAME,
)

rag_adapter = NeoOLAFSemanticRAGAdapter(semantic_rag_engine)

# ---------------------------------------------------------
# Pipeline
# ---------------------------------------------------------
pipeline = Pipeline(
    layers=[
        PreprocessingLayer(
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP,
            translate=TRANSLATE_TO_ENGLISH,
            translator=translator,
            verbose=True,
        ),
        LinguisticExpressionExtractionLayer(
            ollama_backend=openrouter_backend,
            max_chunks=MAX_CHUNKS,
            temperature=0.0,
            verbose=True,
        ),
        CandidateEnrichmentLayer(
            ollama_backend=openrouter_backend,
            wikipedia_source=wikipedia_source,
            wikidata_source=wikidata_source,
            wordnet_source=wordnet_source,
            web_search_source=web_search_source,
            max_expressions=MAX_EXPRESSIONS,
            use_web_search=USE_WEB_SEARCH,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        CandidateTypingResolutionLayer(
            ollama_backend=openrouter_backend,
            max_expressions=MAX_EXPRESSIONS,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        CandidateRelationExtractionLayer(
            ollama_backend=openrouter_backend,
            max_relation_mentions=MAX_RELATION_MENTIONS,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        CandidateTripleGenerationLayer(
            max_assertions=None,
            verbose=True,
        ),
        ConceptRelationInductionLayer(
            ollama_backend=openrouter_backend,
            max_concept_inputs=None,
            max_relation_inputs=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        HierarchisationLayer(
            ollama_backend=openrouter_backend,
            max_concept_pairs=None,
            max_relation_pairs=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        AxiomSchemataExtractionLayer(
            ollama_backend=openrouter_backend,
            max_relation_schema_inputs=None,
            max_subclass_inputs=None,
            temperature=0.0,
            rag_adapter=rag_adapter,
            verbose=True,
        ),
        GeneralAxiomExtractionLayer(
            ollama_backend=openrouter_backend,
            max_schema_inputs=None,
            max_description_inputs=None,
            temperature=0.0,
            verbose=True,
        ),
        ValidationReasoningLayer(
            max_triples=None,
            verbose=True,
        ),
        InferenceCompletionLayer(
            max_inferred_triples=None,
            verbose=True,
        ),
        SerializationLayer(
            output_subdir="data/exports",
            base_uri="http://neoolaf.org/resource/",
            verbose=True,
        ),
    ],
    verbose=True,
    continue_from_last=CONTINUE_FROM_LAST,
)

# ---------------------------------------------------------
# Execution config: process all chunks, then global reasoning
# ---------------------------------------------------------
execution_config = ExecutionConfig(
    mode="chunk_iterative_mode",
    chunk_loop_enabled=True,
    chunk_layer_names=[
        "layer01_linguistic_expression_extraction",
        "layer02_candidate_enrichment",
        "layer03_candidate_typing_resolution",
        "layer04_candidate_relation_extraction",
        "layer05_candidate_triple_generation",
    ],
    global_layer_names=[
        "layer06_concept_relation_induction",
        "layer07_hierarchisation",
        "layer08_axiom_schemata_extraction",
        "layer09_general_axiom_extraction",
        "layer10_validation_reasoning",
        "layer11_inference_completion",
        "layer12_serialization",
    ],
    max_chunks=None,  # run all chunks
)

# ---------------------------------------------------------
# Run
# ---------------------------------------------------------
runner = Runner(
    pipeline=pipeline,
    runs_root="runs",
    verbose=True,
    execution_config=execution_config,
)

final_state = runner.run(state)

# ---------------------------------------------------------
# Inspect
# ---------------------------------------------------------
print("Chunks in document:", len(final_state.document.chunks))
print("Merged linguistic expressions:", len(final_state.linguistic_expressions))
print("Merged enriched expressions:", len(final_state.enriched_expressions))
print("Merged candidate triples:", len(final_state.candidate_triples))
print("Concept candidates:", len(final_state.concept_candidates))
print("Axiom schemata:", len(final_state.axiom_schema_candidates))
print("Completion candidates:", len(final_state.completion_candidates))

print("\nExport folder:")
export_dir = Path(final_state.artifact_dir) / "data" / "exports"
print(export_dir)

if export_dir.exists():
    print("\nExported files:")
    for path in sorted(export_dir.iterdir()):
        print("-", path.name)

[NeoOLAF] Run directory: runs\run_20260407_040006
[NeoOLAF] Execution mode: chunk_iterative_mode
[NeoOLAF] Pipeline started with 1 layers
[NeoOLAF] Layer 0/0: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 93.48s
[NeoOLAF] Pipeline finished in 93.48s
[NeoOLAF] Chunk iterative mode will process 56 chunks
[NeoOLAF] Processing chunk 1/56: chunk_0000
[NeoOLAF] Pipeline started with 5 layers
[NeoOLAF] Layer 0/4: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction


[NeoOLAF] Finished layer: layer01_linguistic_expression_extraction in 28.23s
[NeoOLAF] Layer 1/4: layer02_candidate_enrichment

[NeoOLAF] Starting layer: layer02_candidate_enrichment


[NeoOLAF] Finished layer: layer02_candidate_enrichment in 439.54s
[NeoOLAF] Layer 2/4: layer03_candidate_typing_resolution

[NeoOLAF] Starting layer: layer03_candidate_typing_resolution


KeyboardInterrupt: 